In [9]:
import os, glob, vcf, math
import pandas as pd


def parsing (line):
    CHR, POS, REF, ALT = line[0], int(line[1]), line[3], line[4]

    # 7. info
    info_list = line[7].split(';')
    info_dict = {}
    for i in range( len(info_list) ):
        if "=" in info_list[i]:
            info_dict [info_list[i].split('=')[0]] = info_list[i].split('=')[1]

    # 8 : format
    format_list = line[8].split(":")

    # 9 ~ : sample
    sample_dict = []
    for i in range(9, len(line)):
        temp_dict = {}
        if line[i] == ".":      # 없는 경우
            for j_index, j in enumerate (format_list):
                temp_dict[ format_list[j_index] ] = "."
            sample_dict.append (temp_dict)
        else:
            for j_index, j in enumerate(line[i].split(":")):
                temp_dict[ format_list[j_index] ] = j
            sample_dict.append (temp_dict)

    return CHR, POS, REF, ALT, info_list, info_dict, format_list, sample_dict

def FIND_CONTROL_SAMPLES ( samplenames, str_contain = "Blood" ):
    for i, samplename in enumerate(samplenames):
        if str_contain in samplename:
            return i, samplename

####################

def POPULATION ( VEP_dict, info_dict, **kwargs ):
    for pop in kwargs ["population_db"]:
        if pop in VEP_dict.keys():
            if VEP_dict [ pop ] not in  ["", "."]:
                info_dict [ pop ] = float ( VEP_dict[ pop ] )
                if info_dict [ pop ]  > info_dict [ "PAF_max" ]:
                    info_dict [ "PAF_max" ] = info_dict [ pop ]
    return info_dict

def SPLICEAI ( VEP_dict ):
    import numpy as np
    import math
    filtered_list = [ float(value) for value in [VEP_dict["SpliceAI_pred_DS_AG"], VEP_dict["SpliceAI_pred_DS_AL"],  VEP_dict["SpliceAI_pred_DS_DG"], VEP_dict["SpliceAI_pred_DS_DL"]] if value != '' and value != None]
    if len ( filtered_list ) != 0:
        SpliceAI_score = np.max ( np.array ( filtered_list ) )
    else:
        SpliceAI_score = math.nan
    return SpliceAI_score

def ADD_MATRIX ( record, Variant_ID, Sample_ID, add):
    add_matrix = [ Variant_ID, Sample_ID]
    for samplename in samplenames:
        add_matrix.append ( record.samples [ samplenames_dict_rev[samplename] ].data.AD )
    add_matrix = add_matrix + add
    return add_matrix

### pyvcf를 이용한 parsing

In [10]:
kwargs = { "am_pathogenicity" : 0.56, 
                "Consequence" : ["stop_gained", "frameshift_variant", "splice_donor_variant", "splice_acceptor_variant"], 
                "SpliceAI_score" : 0.4, 
                "population_db" : ["dbSNP_K1", "dbSNP_KRG", "dbSNP_EAS_AF", "gnomAD_AF" ] }


# Sample_IDs = ["250310", "250319", "250407", "250408", "250428", "250509", "250513", "250515", "250520",  "250526_Left", "250526_Right", "250527", "250530",  "250602", "250605",  "250609"]
# Sample_IDs = ["250520", "221021", "231011", "231208", "231220", "240112", "250227", "250617", "250627", "250704", "250716", "250723_Ant", "250723_Post", "250725"]
#Sample_IDs = [  "250526_Left", "250526_Right"]
Sample_IDs = ["SMS"]

df_acc = pd.DataFrame ( columns = [ "Variant_ID", "Sample_ID", "AD", "Consequence", "am_pathogenicity", "SpliceAI_score", "Gene_Symbol", "PAF_max" ])


def ACC_VARIANT ( Sample_IDs, option ):

    for Sample_ID in Sample_IDs:
        for TISSUE in ["multiple"]:

            if option == "mutect":
                VCF_PATH = os.path.join ( "/data/project/Meningioma/04.mutect/02.PASS", f"{Sample_ID}_{TISSUE}.MT2.FMC.HF.RMBLACK.vep.vcf" )
            elif option == "hc":
                VCF_PATH = os.path.join ( "/data/project/Meningioma/06.hc/hg38/04.vep", Sample_ID, "Tumor", f"{Sample_ID}_{TISSUE}.DP100.vep.vcf" )

            if os.path.exists(VCF_PATH):
                print ( VCF_PATH )

                vcf_reader = vcf.Reader(open( VCF_PATH, "r"))
                CSQ_title = vcf_reader.infos["CSQ"].desc.split ("Format: ")[1].split ("|")        # CSQ의 이름 
                global samplenames, samplenames_dict, samplenames_dict_rev
                samplenames = vcf_reader.samples
                samplenames_dict = { i : samplenames [i] for i in range ( len (samplenames ))  }        # {0: '250310_Tumor'}
                samplenames_dict_rev = { samplenames [i] : i for i in range ( len (samplenames ))  } # { '250310_Tumor' : 0}

                blood_index, blood_samplename = FIND_CONTROL_SAMPLES ( samplenames, str_contain = "Blood" )


            
                line_num = 0
                for record in vcf_reader:        # record.CHROM, recrod.POS ,record.ALT
                    CHR, POS, REF, ALT = record.CHROM, 	record.POS, record.REF,  record.ALT

                    # info_dict 초기화
                    info_dict = {}
                    for pop in kwargs ["population_db"]:
                        info_dict [ pop ] = "."
                    info_dict [ "PAF_max"] = 0

                    # df에 덧붙여줄 것
                    global add_df
                    add_df = pd.DataFrame ( columns = df_acc.columns )

                    print (info_dict)

                    # u_list = []
                    # for u_index, u in enumerate( record.INFO["CSQ"] ) :   # Transcript 마다 돌기
                    #     VEP_dict = {}
                    #     for v_index, v in enumerate( u.split ("|") ) :
                    #         VEP_dict [ CSQ_title [v_index] ] = v


                    #     # Gene 정보
                    #     GENE = VEP_dict ["SYMBOL"]

                    #     # Population 정보 (info)dict update
                    #     info_dict = POPULATION ( VEP_dict, info_dict, **kwargs )

                    #     # Splice_AI
                    #     SpliceAI_score = SPLICEAI ( VEP_dict )
                    #     if SpliceAI_score != math.nan:
                    #         if SpliceAI_score > kwargs ["SpliceAI_score"]:
                    #             VEP_dict [ "Consequence" ] = "Splice_Site"
                    #             u_list.append ( u_index )
                    #             add_matrix = ADD_MATRIX ( record, CHR + ":" + str(POS) + "_" + str ( REF[0] ) + ":" + str ( ALT[0] ),Sample_ID,[ VEP_dict ["Consequence"], math.nan, SpliceAI_score, GENE, info_dict ["PAF_max"] ]  )
                    #             add_df.loc [ len(add_df) ] = add_matrix
                    #             #print ( "\t{} : {}\t{}\t{}\t{}".format ( u_index, VEP_dict ["BIOTYPE"], VEP_dict ["Consequence"], VEP_dict ["am_pathogenicity"], SpliceAI_score  ) )


                    #     # Canonical coding region
                    #     if ( VEP_dict ["BIOTYPE"] == "protein_coding" ) & ("missense" in VEP_dict ["Consequence"] ):        # missense_variant 인 경우 AlphaMissense를 평가해본다
                    #         if ( VEP_dict ["am_pathogenicity"] != "" ):
                    #             if ( float ( VEP_dict ["am_pathogenicity"] ) > kwargs ["am_pathogenicity"] ) :
                    #                 u_list.append ( u_index )
                    #                 add_matrix = ADD_MATRIX ( record, CHR + ":" + str(POS) + "_" + str ( REF[0] ) + ":" + str ( ALT[0] ), Sample_ID,[ VEP_dict ["Consequence"], float ( VEP_dict ["am_pathogenicity"] ), math.nan, GENE, info_dict ["PAF_max"] ]  )
                    #                 add_df.loc [ len(add_df) ] = add_matrix
                    #                 #print ( "\t{} : {}\t{}\t{}\t{}".format ( u_index, VEP_dict ["BIOTYPE"], VEP_dict ["Consequence"], VEP_dict ["am_pathogenicity"], VEP_dict [ "SpliceAI_score" ]  ) )
                    #         else:
                    #             if (CHR + ":" + str(POS) == "chr9:107487067"):
                    #                 u_list.append ( u_index )
                    #                 add_matrix = ADD_MATRIX ( record, CHR + ":" + str(POS) + "_" + str ( REF[0] ) + ":" + str ( ALT[0] ),  Sample_ID,[ VEP_dict ["Consequence"], math.nan, math.nan, GENE, info_dict ["PAF_max"] ]  )
                    #                 add_df.loc [ len(add_df) ] = add_matrix

                    #     if ( VEP_dict["BIOTYPE"] == "protein_coding" ) and any( i in VEP_dict["Consequence"] for i in kwargs["Consequence"] ):    # 다른 consequence 중 하나라도 포함되면 추가
                    #         u_list.append ( u_index )
                    #         for i in kwargs["Consequence"]:
                    #             if i in VEP_dict["Consequence"]:                            
                    #                 add_matrix = ADD_MATRIX ( record, CHR + ":" + str(POS) + "_" + str ( REF[0] ) + ":" + str ( ALT[0] ), Sample_ID, [ i, math.nan, math.nan, GENE, info_dict ["PAF_max"] ]  )
                    #         add_df.loc [ len(add_df) ] = add_matrix


                            

                    # if u_list != []:
                    #     add_df ["Consequence"] = pd.Categorical( add_df["Consequence"], categories = ["Splice_Site"] + kwargs ["Consequence"] + ["missense_variant"], ordered=True )
                    #     add_df.sort_values(by = ["Consequence", "am_pathogenicity"] )

                    #     # df에 쌓기  (Choose most impactful transcript)
                    #     df_acc.loc[ len(df_acc) ] = add_df.iloc [0, ]

                    #     line_num += 1

                        
ACC_VARIANT ( Sample_IDs = ["SMS"], option = "mutect" )

/data/project/Meningioma/04.mutect/02.PASS/SMS_multiple.MT2.FMC.HF.RMBLACK.vep.vcf
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KRG': '.', 'dbSNP_EAS_AF': '.', 'gnomAD_AF': '.', 'PAF_max': 0}
{'dbSNP_K1': '.', 'dbSNP_KR

In [7]:
samplenames_dict

{0: 'SMS_1st_Tumor',
 1: 'SMS_2nd_Tumor',
 2: 'SMS_3rd_Tumor',
 3: 'SMS_4th_Tumor',
 4: 'SMS_Blood',
 5: 'SMS_mets_Tumor'}